# Insurance Claims & Portfolio Risk Dashboard
### Python analysis notebook — freMTPL2 (French Motor Third-Party Liability)

**Business scenario.** We are analysts on a motor-insurer's analytics team. Management wants
a clear, descriptive (not pricing/underwriting) view of the portfolio: where claims happen,
where they cost the most, which segments deserve attention, and what to monitor going forward.

**Data.** Public `freMTPL2freq` / `freMTPL2sev` datasets from the CASdatasets R package
(Dutang & Charpentier), downloaded directly from the official GitHub mirror of the package
(`github.com/dutangc/CASdatasets`) and converted from `.rda` to CSV. This is the **real, full
dataset**: 677,991 policies and 26,444 individual claims — no sampling, no synthetic data.

Official docs: https://dutangc.github.io/CASdatasets/reference/freMTPL.html

**Important constraint.** freMTPL2 does **not** include written premium, so **Loss Ratio is
never calculated** in this notebook. The core KPIs are claim frequency, claim severity and
total claim cost, as required by the brief.


## Step 0 — Imports & setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RAW_DIR = "../data/raw"
OUT_DIR = "../data/processed"


## Step 1 — Load the two raw tables

`freq` has **one row per policy** (risk features + claim count + exposure).
`sev` has **one row per individual claim**, linked back to a policy through `IDpol`.
A policy can appear zero, one, or several times in `sev` depending on how many claims it had.

In [2]:
freq = pd.read_csv(f"{RAW_DIR}/freMTPL2freq_raw.csv")
sev = pd.read_csv(f"{RAW_DIR}/freMTPL2sev_raw.csv")

print(f"Frequency table : {freq.shape[0]:,} policies, {freq.shape[1]} columns")
print(f"Severity table  : {sev.shape[0]:,} individual claims, {sev.shape[1]} columns")
freq.head()


Frequency table : 677,991 policies, 12 columns
Severity table  : 26,444 individual claims, 2 columns


,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0,0.0,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3.0,0.0,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5.0,0.0,0.75,6,2,52,50,B12,Diesel,B,54,Picardie
3,10.0,0.0,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine
4,11.0,0.0,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine


In [3]:
sev.head()

,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


## Step 2 — Data quality checks

Before touching a single number we check: missing values, duplicate policies, and whether the
two tables agree with each other (every `ClaimNb > 0` should have exactly that many rows in
`sev`, and every claim in `sev` should belong to a real policy).

In [4]:
print("Missing values in freq:", freq.isna().sum().sum())
print("Missing values in sev :", sev.isna().sum().sum())
print("Duplicate policy IDs  :", freq["IDpol"].duplicated().sum())

sev_claim_counts = sev.groupby("IDpol").size().rename("sev_claim_count")
consistency = (
    freq.set_index("IDpol")[["ClaimNb"]]
    .join(sev_claim_counts, how="left")
    .fillna(0)
)
mismatches = (consistency["ClaimNb"] != consistency["sev_claim_count"]).sum()
print(f"Policies where ClaimNb disagrees with the severity-table row count: {mismatches}")


Missing values in freq: 0
Missing values in sev : 0
Duplicate policy IDs  : 0
Policies where ClaimNb disagrees with the severity-table row count: 0


### 2.1 Known, documented data-entry issues

freMTPL2 is a widely studied public dataset, and two data-entry artefacts are well documented
in the actuarial literature (e.g. Wüthrich's case studies using this same data):

1. **`Exposure` should never exceed 1.0** — it is a fraction of the one-year observation
   window. A small number of rows (1,224) go up to ~2.01. We **cap** `Exposure` at 1.0.
2. **A handful of policies (5) report implausible claim counts** (8–16 claims within well
   under a year of exposure), all sharing the same region/density cell — a clear encoding
   artefact rather than genuine behaviour. We **cap** `ClaimNb` at 4 (a threshold also used in
   published work on this dataset).

Both are **corrections of extreme/implausible values, not deletion of rows** — in line with the
brief's instruction not to silently delete outliers.

In [5]:
n_exposure_capped = (freq["Exposure"] > 1).sum()
n_claimnb_capped = (freq["ClaimNb"] > 4).sum()
print(f"Rows with Exposure > 1 (capped to 1.0): {n_exposure_capped}")
print(f"Rows with ClaimNb  > 4 (capped to 4)  : {n_claimnb_capped}")

freq["Exposure"] = freq["Exposure"].clip(upper=1.0)
freq["ClaimNb"] = freq["ClaimNb"].clip(upper=4)


Rows with Exposure > 1 (capped to 1.0): 1224
Rows with ClaimNb  > 4 (capped to 4)  : 8


### 2.2 VehAge placeholder codes

`VehAge` contains 48 rows with values 99 or 100 — almost certainly an "unknown" placeholder
code, not a 99-year-old car. We keep these rows and simply let the vehicle-age bucketing below
fold them into the open-ended "16+ years" segment, where they belong regardless of the exact
code.

## Step 3 — Aggregate claim cost per policy, then merge

**This is the documented merge logic required by the brief.** A policy can have several
claims, so we first aggregate `sev` to one row per `IDpol` (sum of claim amounts, and a count
of how many claim records exist), and only then left-merge onto `freq`. This guarantees the
final table has exactly one row per policy — the correct grain for portfolio KPIs — while
still preserving every euro of claim cost.

In [6]:
policy_claim_cost = (
    sev.groupby("IDpol", as_index=False)
    .agg(ClaimAmount_sum=("ClaimAmount", "sum"),
         ClaimAmount_count=("ClaimAmount", "count"))
)

df = freq.merge(policy_claim_cost, on="IDpol", how="left")
df["ClaimAmount_sum"] = df["ClaimAmount_sum"].fillna(0.0)
df["ClaimAmount_count"] = df["ClaimAmount_count"].fillna(0).astype(int)

# Sanity check: no cost lost or duplicated in the merge
assert np.isclose(df["ClaimAmount_sum"].sum(), sev["ClaimAmount"].sum())
print(f"Policy-level table after merge: {df.shape[0]:,} rows (1 row per policy)")
df.head()


Policy-level table after merge: 677,991 rows (1 row per policy)


,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,ClaimAmount_sum,ClaimAmount_count
0,1.0,0.0,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,0.0,0
1,3.0,0.0,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,0.0,0
2,5.0,0.0,0.75,6,2,52,50,B12,Diesel,B,54,Picardie,0.0,0
3,10.0,0.0,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine,0.0,0
4,11.0,0.0,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine,0.0,0


## Step 4 — Build business segments

We turn continuous risk features into readable bands for segment comparison and for Tableau
filters. Bin edges are chosen to be business-sensible (e.g. legal driving age starts at 18,
new-vehicle vs. old-vehicle cutoffs, the bonus-malus "50" baseline for a claim-free driver in
the French system) rather than purely statistical quantiles.

In [7]:
df["DrivAgeBand"] = pd.cut(
    df["DrivAge"], bins=[17, 25, 35, 45, 55, 65, 75, 101],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76+"],
)
df["VehAgeBand"] = pd.cut(
    df["VehAge"], bins=[-1, 1, 5, 10, 15, 200],
    labels=["0-1", "2-5", "6-10", "11-15", "16+"],
)
df["VehPowerBand"] = pd.cut(
    df["VehPower"], bins=[0, 6, 9, 20],
    labels=["Low (<=6)", "Mid (7-9)", "High (10+)"],
)
df["BonusMalusBand"] = pd.cut(
    df["BonusMalus"], bins=[0, 50, 100, 150, 300],
    labels=["50 (base/no-claims)", "51-100", "101-150", "151+"],
)
df[["DrivAgeBand", "VehAgeBand", "VehPowerBand", "BonusMalusBand"]].describe()


,DrivAgeBand,VehAgeBand,VehPowerBand,BonusMalusBand
count,677991,677991,677991,677991
unique,7,5,3,4
top,36-45,2-5,Low (<=6),50 (base/no-claims)
freq,170345,191610,389131,384142


## Step 5 — Core KPI function

One function, used everywhere below, so every table in this notebook (and every chart in
Tableau) uses **exactly the same KPI definitions** — no silent inconsistency between the
portfolio summary and the segment tables.

- **Claim Frequency = Claim Count / Exposure** (claims per policy-year — this is what makes
  segments with different amounts of exposure comparable).
- **Average Claim Severity = Total Claim Cost / number of claim records** (average cost
  *per claim*, not per policy).

In [8]:
def kpi_table(frame: pd.DataFrame, group_cols=None) -> pd.DataFrame:
    grouped = frame if group_cols is None else frame.groupby(group_cols, observed=True)

    def _agg(g):
        policy_count = len(g)
        exposure = g["Exposure"].sum()
        claim_count = g["ClaimNb"].sum()
        total_cost = g["ClaimAmount_sum"].sum()
        frequency = claim_count / exposure if exposure > 0 else np.nan
        n_claim_records = g["ClaimAmount_count"].sum()
        severity = total_cost / n_claim_records if n_claim_records > 0 else np.nan
        return pd.Series({
            "PolicyCount": policy_count, "Exposure": exposure, "ClaimCount": claim_count,
            "ClaimFrequency": frequency, "TotalClaimCost": total_cost,
            "AvgClaimSeverity": severity,
        })

    if group_cols is None:
        return _agg(frame).to_frame().T
    return grouped.apply(_agg, include_groups=False).reset_index()


## Q1 — What is the overall health of the insurance portfolio?

Raw claim counts alone are misleading because portfolios differ in size and in how long each
policy was observed (`Exposure`). Two portfolios with the same number of claims can have very
different underlying risk if one has twice the exposure. **Claim Frequency** (claims per
policy-year) fixes this by normalizing for exposure, which is why it — not the raw claim count
— is the headline risk KPI.

In [9]:
q1_portfolio = kpi_table(df)
q1_portfolio.round(2)


,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,677991.0,358343.5,26405.0,0.07,59909216.5,2265.51


## Q2 — Which regions generate the greatest claim burden?

We deliberately do **not** rank regions by raw claim count. A large region simply has more
policies, so of course it has more claims — that says nothing about risk. Instead we compare
**exposure, claim frequency, total claim cost and average severity together**.

In [10]:
q2_region = kpi_table(df, "Region").sort_values("TotalClaimCost", ascending=False)
q2_region["CostPerPolicy"] = q2_region["TotalClaimCost"] / q2_region["PolicyCount"]
q2_region["CostPerExposureYear"] = q2_region["TotalClaimCost"] / q2_region["Exposure"]
q2_region.round(2)


,Region,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity,CostPerPolicy,CostPerExposureYear
6,Centre,160595.0,102701.39,6475.0,0.06,19074802.36,2945.92,118.78,185.73
21,Rhone-Alpes,84748.0,45328.72,4233.0,0.09,10278303.85,2428.14,121.28,226.75
20,Provence-Alpes-Cotes-D'Azur,79315.0,35748.98,2985.0,0.08,6871024.80,2301.08,86.63,192.20
11,Ile-de-France,69789.0,30196.65,2591.0,0.09,4563221.86,1761.18,65.39,151.12
5,Bretagne,42120.0,27751.77,1871.0,0.07,3901051.25,2085.01,92.62,140.57
17,Pays-de-la-Loire,38747.0,21927.44,1576.0,0.07,2570964.93,1631.32,66.35,117.25
12,Languedoc-Roussillon,35805.0,14710.62,1030.0,0.07,2030911.08,1901.60,56.72,138.06
1,Aquitaine,31329.0,14316.17,1055.0,0.07,1916138.54,1816.25,61.16,133.84
16,Nord-Pas-de-Calais,27284.0,11487.07,944.0,0.08,1634689.72,1731.66,59.91,142.31
19,Poitou-Charentes,19046.0,11163.01,800.0,0.07,1305107.78,1631.38,68.52,116.91


## Q3 — Which driver and vehicle segments show different claim patterns?

We compare frequency and severity across driver-age band, vehicle-age band, vehicle-power
band, fuel type and bonus-malus band. Patterns are described, **not** interpreted causally —
these are portfolio associations, not proof that (say) being young *causes* more claims.

In [11]:
q3_drivage = kpi_table(df, "DrivAgeBand")
q3_vehage = kpi_table(df, "VehAgeBand")
q3_vehpower = kpi_table(df, "VehPowerBand")
q3_vehgas = kpi_table(df, "VehGas")
q3_bonusmalus = kpi_table(df, "BonusMalusBand")
q3_drivage.round(3)


,DrivAgeBand,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,18-25,38893.0,16225.403,2403.0,0.148,12307261.54,5121.624
1,26-35,149180.0,69236.758,5167.0,0.075,10607248.80,2052.883
2,36-45,170345.0,87653.056,6315.0,0.072,12119011.84,1919.083
3,46-55,161895.0,89228.796,6584.0,0.074,12326494.29,1861.446
4,56-65,90687.0,51755.373,3316.0,0.064,6267615.63,1890.113
5,66-75,48047.0,30755.473,1769.0,0.058,4104039.55,2318.666
6,76+,18944.0,13488.635,851.0,0.063,2177544.85,2558.807


In [12]:
q3_vehage.round(3)

,VehAgeBand,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,0-1,129018.0,49434.336,3509.0,0.071,7485825.24,2133.322
1,2-5,191610.0,101717.803,7582.0,0.075,15366736.33,2026.739
2,6-10,171587.0,98968.616,8008.0,0.081,15805071.87,1973.414
3,11-15,134059.0,77952.015,5642.0,0.072,17808346.08,3135.272
4,16+,51717.0,30270.725,1664.0,0.055,3443236.98,2069.253


In [13]:
q3_vehpower.round(3)

,VehPowerBand,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,Low (<=6),389131.0,210688.816,15385.0,0.073,32633777.40,2115.916
1,Mid (7-9),222436.0,115928.567,8457.0,0.073,21646115.32,2559.247
2,High (10+),66424.0,31726.113,2563.0,0.081,5629323.78,2196.381


In [14]:
q3_vehgas.round(3)

,VehGas,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,Diesel,332128.0,170580.633,13449.0,0.079,27594673.95,2051.649
1,Regular,345863.0,187762.862,12956.0,0.069,32314542.55,2486.882


In [15]:
q3_bonusmalus.round(3)

,BonusMalusBand,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity
0,50 (base/no-claims),384142.0,225142.011,11553.0,0.051,22282394.02,1922.222
1,51-100,286055.0,129630.756,13596.0,0.105,35066892.92,2579.207
2,101-150,7585.0,3475.601,1202.0,0.346,2427253.15,2019.345
3,151+,209.0,95.128,54.0,0.568,132676.41,2456.971


## Q4 — Are frequent-claim segments also expensive-claim segments?

We build a driver-age x vehicle-power grid (dropping tiny cells with under 20 claims, which
would be too noisy to trust) and classify each cell into a frequency/severity quadrant relative
to the median of the grid. This distinguishes "claims often but cheaply" segments from
"claims rarely but expensively" segments — a distinction a single KPI can never show.

In [16]:
q4_quadrant = kpi_table(df, ["DrivAgeBand", "VehPowerBand"])
q4_quadrant = q4_quadrant[q4_quadrant["ClaimCount"] >= 20]

freq_median = q4_quadrant["ClaimFrequency"].median()
sev_median = q4_quadrant["AvgClaimSeverity"].median()
q4_quadrant["FrequencyLevel"] = np.where(q4_quadrant["ClaimFrequency"] >= freq_median, "High", "Low")
q4_quadrant["SeverityLevel"] = np.where(q4_quadrant["AvgClaimSeverity"] >= sev_median, "High", "Low")
q4_quadrant["Quadrant"] = q4_quadrant["FrequencyLevel"] + " Frequency / " + q4_quadrant["SeverityLevel"] + " Severity"
q4_quadrant.round(3)


,DrivAgeBand,VehPowerBand,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity,FrequencyLevel,SeverityLevel,Quadrant
0,18-25,Low (<=6),27084.0,11540.022,1608.0,0.139,6030163.96,3750.102,High,High,High Frequency / High Severity
1,18-25,Mid (7-9),10210.0,4088.594,696.0,0.170,6123141.19,8797.617,High,High,High Frequency / High Severity
2,18-25,High (10+),1599.0,596.788,99.0,0.166,153956.39,1555.115,High,Low,High Frequency / Low Severity
3,26-35,Low (<=6),91712.0,43752.510,3114.0,0.071,6511893.19,2091.167,Low,High,Low Frequency / High Severity
4,26-35,Mid (7-9),45651.0,20489.807,1593.0,0.078,2993888.26,1879.403,High,Low,High Frequency / Low Severity
5,26-35,High (10+),11817.0,4994.441,460.0,0.092,1101467.35,2394.494,High,High,High Frequency / High Severity
6,36-45,Low (<=6),91753.0,48553.418,3462.0,0.071,5698944.17,1646.142,High,Low,High Frequency / Low Severity
7,36-45,Mid (7-9),59531.0,30162.653,2123.0,0.070,4742730.17,2233.976,Low,High,Low Frequency / High Severity
8,36-45,High (10+),19061.0,8936.985,730.0,0.082,1677337.50,2297.723,High,High,High Frequency / High Severity
9,46-55,Low (<=6),89614.0,50557.452,3762.0,0.074,7168678.66,1886.494,High,Low,High Frequency / Low Severity


In [17]:
q4_quadrant["Quadrant"].value_counts()

Quadrant
High Frequency / Low Severity     6
Low Frequency / High Severity     6
High Frequency / High Severity    5
Low Frequency / Low Severity      4
Name: count, dtype: int64

## Q5 — Is claim cost concentrated in a small number of claims?

A Pareto-style analysis on the raw, claim-level severity table (every one of the 26,444
individual claims, ranked by size).

In [18]:
pareto = sev.sort_values("ClaimAmount", ascending=False).reset_index(drop=True)
pareto["ClaimRank"] = pareto.index + 1
pareto["CumulativeCost"] = pareto["ClaimAmount"].cumsum()
total_cost_all_claims = pareto["ClaimAmount"].sum()
pareto["CumulativeCostShare"] = pareto["CumulativeCost"] / total_cost_all_claims
pareto["ClaimRankShare"] = pareto["ClaimRank"] / len(pareto)

def share_of_cost_from_top_n_pct(pct):
    n = int(len(pareto) * pct)
    return pareto.iloc[:n]["ClaimAmount"].sum() / total_cost_all_claims

for pct in [0.01, 0.05, 0.10, 0.20]:
    print(f"Top {pct:>4.0%} of claims by size generate {share_of_cost_from_top_n_pct(pct):.1%} of total claim cost")


Top   1% of claims by size generate 38.0% of total claim cost
Top   5% of claims by size generate 52.1% of total claim cost
Top  10% of claims by size generate 59.9% of total claim cost
Top  20% of claims by size generate 68.8% of total claim cost


In [19]:
# Downsample to ~200 points so the Tableau file stays light without losing the curve's shape
step = max(1, len(pareto) // 200)
pareto_curve = pareto.iloc[::step][
    ["ClaimRank", "ClaimRankShare", "ClaimAmount", "CumulativeCost", "CumulativeCostShare"]
].copy()
pareto_curve.head()


,ClaimRank,ClaimRankShare,ClaimAmount,CumulativeCost,CumulativeCostShare
0,1,0.000038,4075400.56,4075400.56,0.068026
132,133,0.005029,34463.16,19743176.05,0.329552
264,265,0.010021,16510.22,22773883.74,0.380140
396,397,0.015013,11715.01,24602629.73,0.410665
528,529,0.020005,9743.71,25982222.69,0.433693


This concentration matters for **claims management**: a small team focused on reviewing and
actively managing the largest ~5–10% of claims can influence the majority of total claim cost —
far more leverage than spreading effort evenly across all claims.

## Q6 — What unusual claims or segments should management investigate?

**We never auto-delete large claims.** Instead we flag them for human review using an IQR rule
on log-cost (claim costs are heavily right-skewed, so raw-scale IQR would be meaningless), and
separately look for thin Region x Area cells with frequency far above the portfolio average.

In [20]:
log_cost = np.log1p(sev["ClaimAmount"])
q1_, q3_ = log_cost.quantile([0.25, 0.75])
iqr = q3_ - q1_
upper_fence = q3_ + 3 * iqr
extreme_mask = log_cost > upper_fence

extreme_claims = sev.loc[extreme_mask].merge(
    df[["IDpol", "Region", "VehBrand", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Area", "VehGas"]],
    on="IDpol", how="left",
).sort_values("ClaimAmount", ascending=False)

print(f"Claims flagged for review by the statistical rule: {len(extreme_claims)} "
      f"({len(extreme_claims)/len(sev):.1%} of all claims, "
      f"{extreme_claims['ClaimAmount'].sum()/sev['ClaimAmount'].sum():.1%} of total cost)")
extreme_claims.head(15)


Claims flagged for review by the statistical rule: 878 (3.3% of all claims, 48.0% of total cost)


,IDpol,ClaimAmount,Region,VehBrand,VehPower,VehAge,DrivAge,BonusMalus,Area,VehGas
338,1120377,4075400.56,Centre,B2,9,13,19,100,B,Regular
520,110846,1403057.40,Centre,B1,6,13,20,100,C,Regular
392,2141337,1301172.60,Rhone-Alpes,B2,4,14,18,100,D,Regular
577,3122016,774411.50,Rhone-Alpes,B11,7,7,40,63,E,Diesel
161,2008127,390742.27,Rhone-Alpes,B4,4,2,57,50,D,Regular
178,3025890,369131.88,Champagne-Ardenne,B12,7,1,36,50,A,Diesel
341,1117644,307096.42,Bretagne,B3,7,1,21,90,C,Diesel
224,158309,301635.49,Centre,B1,6,3,46,50,A,Diesel
606,3075820,287423.00,Provence-Alpes-Cotes-D'Azur,B10,7,18,78,50,E,Regular
589,3150210,281897.49,Centre,B3,6,4,61,54,D,Diesel


In [21]:
region_area_cells = kpi_table(df, ["Region", "Area"])
region_area_cells = region_area_cells[region_area_cells["Exposure"] >= 50]
portfolio_freq = q1_portfolio["ClaimFrequency"].iloc[0]
region_area_cells["FreqVsPortfolio"] = region_area_cells["ClaimFrequency"] / portfolio_freq
unusual_cells = region_area_cells.sort_values("FreqVsPortfolio", ascending=False).head(10)
unusual_cells.round(3)


,Region,Area,PolicyCount,Exposure,ClaimCount,ClaimFrequency,TotalClaimCost,AvgClaimSeverity,FreqVsPortfolio
93,Picardie,E,1802.0,847.414,93.0,0.110,147671.05,1587.861,1.489
108,Rhone-Alpes,E,26239.0,13883.219,1503.0,0.108,3842867.55,2556.798,1.469
34,Centre,E,9380.0,4782.335,503.0,0.105,978160.49,1944.653,1.427
107,Rhone-Alpes,D,23012.0,12197.147,1261.0,0.103,4019178.08,3187.294,1.403
103,Provence-Alpes-Cotes-D'Azur,E,27238.0,13261.114,1348.0,0.102,3230581.23,2394.797,1.380
67,Limousin,C,1407.0,768.380,76.0,0.099,101666.50,1337.717,1.342
97,Poitou-Charentes,D,5471.0,3104.214,305.0,0.098,425878.78,1396.324,1.333
59,Ile-de-France,F,17953.0,8124.954,774.0,0.095,1179606.42,1524.039,1.293
29,Bretagne,E,4121.0,2389.429,225.0,0.094,333786.89,1483.497,1.278
83,Nord-Pas-de-Calais,E,7044.0,2582.248,243.0,0.094,452127.68,1860.608,1.277


## Q7 & Q8 — Dashboard KPIs and recommendations

Answered in full in `report/business_summary.md` and in the Tableau build guide
(`tableau/dashboard_build_guide.md`), using the tables produced below.

## Step 6 — Export clean, Tableau-ready tables

Everything above is exported as flat CSV files under `data/processed/`. The main file
(`tableau_portfolio.csv`) is policy-level (1 row per policy) so Tableau can filter, drill down
and re-aggregate live; the KPI tables are pre-aggregated for fast-loading dashboard cards.

In [22]:
tableau_main_cols = [
    "IDpol", "Region", "Area", "Density", "VehBrand", "VehGas",
    "VehPower", "VehPowerBand", "VehAge", "VehAgeBand",
    "DrivAge", "DrivAgeBand", "BonusMalus", "BonusMalusBand",
    "Exposure", "ClaimNb", "ClaimAmount_sum", "ClaimAmount_count",
]
df[tableau_main_cols].rename(columns={"ClaimAmount_sum": "ClaimAmount"}).to_csv(
    f"{OUT_DIR}/tableau_portfolio.csv", index=False)

q1_portfolio.to_csv(f"{OUT_DIR}/kpi_portfolio_summary.csv", index=False)
q2_region.to_csv(f"{OUT_DIR}/kpi_region_summary.csv", index=False)
q3_drivage.to_csv(f"{OUT_DIR}/kpi_segment_driver_age.csv", index=False)
q3_vehage.to_csv(f"{OUT_DIR}/kpi_segment_vehicle_age.csv", index=False)
q3_vehpower.to_csv(f"{OUT_DIR}/kpi_segment_vehicle_power.csv", index=False)
q3_vehgas.to_csv(f"{OUT_DIR}/kpi_segment_fuel_type.csv", index=False)
q3_bonusmalus.to_csv(f"{OUT_DIR}/kpi_segment_bonus_malus.csv", index=False)
q4_quadrant.to_csv(f"{OUT_DIR}/kpi_frequency_severity_quadrant.csv", index=False)
pareto_curve.to_csv(f"{OUT_DIR}/pareto_claim_cost_curve.csv", index=False)
extreme_claims.to_csv(f"{OUT_DIR}/extreme_claims_for_review.csv", index=False)
unusual_cells.to_csv(f"{OUT_DIR}/unusual_region_area_cells.csv", index=False)

print("All Tableau-ready CSV files written to data/processed/")


All Tableau-ready CSV files written to data/processed/
